In [ ]:
# Verificar GPU de Colab
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [1]:
# 1. Configurar GPU y verificar recursos
import tensorflow as tf
import time

print(f"TensorFlow: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU disponible: {gpus}")

if gpus:
    # Obtener info de GPU
    !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
    
    # Habilitar memory growth para evitar OOM
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    
    # Detectar tipo de GPU
    gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_name = gpu_name[0] if gpu_name else "Unknown"
    
    if "A100" in gpu_name:
        print("🚀 A100 detectada - Colab Pro+ (entrenamiento ~15-20 min)")
        BATCH_SIZE = 64
        PREFETCH = tf.data.AUTOTUNE
    elif "V100" in gpu_name:
        print("⚡ V100 detectada - Colab Pro (entrenamiento ~25-35 min)")
        BATCH_SIZE = 48
        PREFETCH = tf.data.AUTOTUNE
    else:  # T4
        print("🔧 T4 detectada - Colab Free (entrenamiento ~60-90 min)")
        BATCH_SIZE = 32
        PREFETCH = 2
else:
    print("⚠️ Sin GPU - entrenamiento muy lento")
    BATCH_SIZE = 16
    PREFETCH = 1

print(f"\nBatch size configurado: {BATCH_SIZE}")

TensorFlow: 2.19.0
GPU disponible: []
⚠️ Sin GPU - entrenamiento muy lento

Batch size configurado: 16


In [2]:
# 2. Instalar dependencias
!pip install -q tf2onnx onnx albumentations

import numpy as np
import os
import json
from datetime import datetime

# Configuración de entrenamiento robusto
CONFIG = {
    'IMG_SIZE': 224,
    'NUM_CLASSES': 38,
    'EPOCHS_PHASE1': 15,      # Más epochs para clasificador
    'EPOCHS_PHASE2': 25,      # Fine-tuning más largo
    'LEARNING_RATE_1': 1e-3,  # LR inicial para clasificador
    'LEARNING_RATE_2': 1e-5,  # LR bajo para fine-tuning
    'WARMUP_EPOCHS': 3,       # Warmup gradual
    'LABEL_SMOOTHING': 0.1,   # Reduce overfitting
    'DROPOUT_RATE': 0.4,      # Más regularización
    'MIXUP_ALPHA': 0.2,       # Data augmentation avanzada
}

print("Configuración de entrenamiento:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.8/455.8 kB 8.4 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 69.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 69.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", bu

In [3]:
# 3. Descargar dataset PlantVillage
import os
import urllib.request
import zipfile

DATASET_URL = "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip"
DATASET_ZIP = "plantvillage.zip"
DATA_DIR = "PlantVillage-Dataset-master/raw/color"

if not os.path.exists("PlantVillage-Dataset-master"):
    print("📥 Descargando dataset PlantVillage (~250MB)...")
    start = time.time()
    urllib.request.urlretrieve(DATASET_URL, DATASET_ZIP)
    print(f"✅ Descargado en {time.time()-start:.1f}s")
    
    print("📦 Extrayendo...")
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zip_ref:
        zip_ref.extractall(".")
    os.remove(DATASET_ZIP)
    print("✅ Dataset listo!")
else:
    print("✅ Dataset ya existe")

# Verificar estructura
classes = sorted(os.listdir(DATA_DIR))
print(f"\n📊 {len(classes)} clases encontradas:")
for i, c in enumerate(classes[:5]):
    count = len(os.listdir(os.path.join(DATA_DIR, c)))
    print(f"  {i}: {c} ({count} imgs)")
print(f"  ...")

total_images = sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in classes)
print(f"\n📷 Total: {total_images:,} imágenes")

📥 Descargando dataset PlantVillage (~250MB)...
✅ Descargado en 68.5s
📦 Extrayendo...
✅ Descargado en 68.5s
📦 Extrayendo...
✅ Dataset listo!

📊 38 clases encontradas:
  0: Apple___Apple_scab (630 imgs)
  1: Apple___Black_rot (621 imgs)
  2: Apple___Cedar_apple_rust (275 imgs)
  3: Apple___healthy (1645 imgs)
  4: Blueberry___healthy (1502 imgs)
  ...

📷 Total: 54,305 imágenes
✅ Dataset listo!

📊 38 clases encontradas:
  0: Apple___Apple_scab (630 imgs)
  1: Apple___Black_rot (621 imgs)
  2: Apple___Cedar_apple_rust (275 imgs)
  3: Apple___healthy (1645 imgs)
  4: Blueberry___healthy (1502 imgs)
  ...

📷 Total: 54,305 imágenes


In [4]:
# 4. Data augmentation agresiva + generadores optimizados
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import albumentations as A

IMG_SIZE = CONFIG['IMG_SIZE']

# Augmentation más agresiva para mejor generalización
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,           # Más rotación
    width_shift_range=0.3,       # Más desplazamiento
    height_shift_range=0.3,
    shear_range=0.2,             # Nuevo: cizallamiento
    zoom_range=0.3,              # Zoom
    horizontal_flip=True,
    vertical_flip=True,          # Nuevo: flip vertical
    brightness_range=[0.7, 1.3], # Nuevo: variación de brillo
    fill_mode='reflect',         # Mejor que 'nearest'
    validation_split=0.15        # 15% validación
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.15
)

print(f"🔄 Creando generadores con batch_size={BATCH_SIZE}...")

train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_gen = val_datagen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

NUM_CLASSES = len(train_gen.class_indices)
STEPS_PER_EPOCH = train_gen.samples // BATCH_SIZE
VAL_STEPS = val_gen.samples // BATCH_SIZE

print(f"\n📊 Dataset split:")
print(f"  Training: {train_gen.samples:,} imágenes ({STEPS_PER_EPOCH} steps/epoch)")
print(f"  Validation: {val_gen.samples:,} imágenes ({VAL_STEPS} steps)")
print(f"  Clases: {NUM_CLASSES}")

🔄 Creando generadores con batch_size=16...
Found 46176 images belonging to 38 classes.
Found 46176 images belonging to 38 classes.
Found 8129 images belonging to 38 classes.
Found 8129 images belonging to 38 classes.

📊 Dataset split:
  Training: 46,176 imágenes (2886 steps/epoch)
  Validation: 8,129 imágenes (508 steps)
  Clases: 38

📊 Dataset split:
  Training: 46,176 imágenes (2886 steps/epoch)
  Validation: 8,129 imágenes (508 steps)
  Clases: 38


In [5]:
# 5. Crear modelo MobileNetV2 optimizado para producción
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

# Base model con pesos de ImageNet
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# Congelar todas las capas inicialmente
base_model.trainable = False

# Cabeza de clasificación más robusta
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)                    # Normalización
x = Dense(512, activation='relu', 
          kernel_regularizer=l2(0.01))(x)      # Más neuronas + L2
x = Dropout(CONFIG['DROPOUT_RATE'])(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu',
          kernel_regularizer=l2(0.01))(x)
x = Dropout(CONFIG['DROPOUT_RATE'])(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

print(f"🏗️ Modelo creado:")
print(f"  Parámetros totales: {model.count_params():,}")
print(f"  Parámetros entrenables: {sum(tf.keras.backend.count_params(w) for w in model.trainable_weights):,}")
print(f"  Capas base: {len(base_model.layers)}")
print(f"  Dropout: {CONFIG['DROPOUT_RATE']}")

# Mostrar arquitectura resumida
model.summary(show_trainable=True, expand_nested=False)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
🏗️ Modelo creado:
  Parámetros totales: 3,062,118
  Parámetros entrenables: 800,550
  Capas base: 154
  Dropout: 0.4
🏗️ Modelo creado:
  Parámetros totales: 3,062,118
  Parámetros entrenables: 800,550
  Capas base: 154
  Dropout: 0.4


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)      ┃ Output Shape    ┃   Param # ┃ Connected to   ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━┩
│ input_layer       │ (None, 224,     │         0 │ -              │   -   │
│ (InputLayer)      │ 224, 3)         │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ Conv1 (Conv2D)    │ (None, 112,     │       864 │ input_layer[0… │   N   │
│                   │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ bn_Conv1          │ (None, 112,     │       128 │ Conv1[0][0]    │   N   │
│ (BatchNormalizat… │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ Conv1_relu (ReLU) │ (None, 112,     │         0 │ bn_Conv1[0][0] │   -   │
│                   │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ expanded_conv_de… │ (None, 112,     │       288 │ Conv1_relu[0]… │   N   │
│ (DepthwiseConv2D) │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ expanded_conv_de… │ (None, 112,     │       128 │ expanded_conv… │   N   │
│ (BatchNormalizat… │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ expanded_conv_de… │ (None, 112,     │         0 │ expanded_conv… │   -   │
│ (ReLU)            │ 112, 32)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ expanded_conv_pr… │ (None, 112,     │       512 │ expanded_conv… │   N   │
│ (Conv2D)          │ 112, 16)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ expanded_conv_pr… │ (None, 112,     │        64 │ expanded_conv… │   N   │
│ (BatchNormalizat… │ 112, 16)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block_1_expand    │ (None, 112,     │     1,536 │ expanded_conv… │   N   │
│ (Conv2D)          │ 112, 96)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block_1_expand_BN │ (None, 112,     │       384 │ block_1_expan… │   N   │
│ (BatchNormalizat… │ 112, 96)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block_1_expand_r… │ (None, 112,     │         0 │ block_1_expan… │   -   │
│ (ReLU)            │ 112, 96)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block_1_pad       │ (None, 113,     │         0 │ block_1_expan… │   -   │
│ (ZeroPadding2D)   │ 113, 96)        │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block_1_depthwise │ (None, 56, 56,  │       864 │ block_1_pad[0… │   N   │
│ (DepthwiseConv2D) │ 96)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block_1_depthwis… │ (None, 56, 56,  │       384 │ block_1_depth… │   N   │
│ (BatchNormalizat… │ 96)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block_1_depthwis… │ (None, 56, 56,  │         0 │ block_1_depth… │   -   │
│ (ReLU)            │ 96)             │           │                │       │
├───────────────────┼─────────────────┼───────────┼────────────────┼───────┤
│ block_1_project   │ (None, 56, 56,  │     2,304 │ block_1_depth… │   N 

 Total params: 3,062,118 (11.68 MB)

 Trainable params: 800,550 (3.05 MB)

 Non-trainable params: 2,261,568 (8.63 MB)

In [6]:
# 6. FASE 1: Entrenar solo cabeza clasificadora con warmup
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, 
    TensorBoard, LearningRateScheduler
)
from tensorflow.keras.losses import CategoricalCrossentropy

# Loss con label smoothing para mejor generalización
loss_fn = CategoricalCrossentropy(label_smoothing=CONFIG['LABEL_SMOOTHING'])

# Compilar con learning rate inicial
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE_1']),
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

# Callbacks para Fase 1
callbacks_phase1 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_phase1.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print("=" * 60)
print("🚀 FASE 1: Entrenando clasificador (base congelada)")
print("=" * 60)
print(f"  Epochs: {CONFIG['EPOCHS_PHASE1']}")
print(f"  Learning rate: {CONFIG['LEARNING_RATE_1']}")
print(f"  Label smoothing: {CONFIG['LABEL_SMOOTHING']}")
print()

start_time = time.time()

history1 = model.fit(
    train_gen,
    epochs=CONFIG['EPOCHS_PHASE1'],
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    verbose=1
)

phase1_time = time.time() - start_time
phase1_acc = max(history1.history['val_accuracy'])
phase1_top3 = max(history1.history['val_top3_acc'])

print(f"\n✅ Fase 1 completada en {phase1_time/60:.1f} minutos")
print(f"   Best Val Accuracy: {phase1_acc:.2%}")
print(f"   Best Val Top-3 Accuracy: {phase1_top3:.2%}")

🚀 FASE 1: Entrenando clasificador (base congelada)
  Epochs: 15
  Learning rate: 0.001
  Label smoothing: 0.1



/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15
 559/2886 ━━━━━━━━━━━━━━━━━━━━ 29:59 773ms/step - accuracy: 0.4456 - loss: 10.0975 - top3_acc: 0.6283

KeyboardInterrupt: 

In [ ]:
# 7. FASE 2: Fine-tuning gradual (descongelar capas progresivamente)
print("=" * 60)
print("🔧 FASE 2: Fine-tuning con descongelado gradual")
print("=" * 60)

# Cargar mejor modelo de fase 1
model.load_weights('best_phase1.keras')

# Estrategia: descongelar últimas 50 capas (de 155 total)
# Esto incluye los últimos 3 bloques de MobileNetV2
base_model.trainable = True
FREEZE_UNTIL = 100  # Congelar primeras 100 capas

for layer in base_model.layers[:FREEZE_UNTIL]:
    layer.trainable = False

trainable_count = sum(1 for layer in model.layers if layer.trainable)
trainable_params = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)

print(f"  Capas descongeladas: {len(base_model.layers) - FREEZE_UNTIL}")
print(f"  Parámetros entrenables: {trainable_params:,}")

# Recompilar con LR muy bajo para fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=CONFIG['LEARNING_RATE_2']),
    loss=loss_fn,
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

# Callbacks para Fase 2 - más pacientes
callbacks_phase2 = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'best_phase2.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print(f"\n  Epochs: {CONFIG['EPOCHS_PHASE2']}")
print(f"  Learning rate: {CONFIG['LEARNING_RATE_2']}")
print()

start_time = time.time()

history2 = model.fit(
    train_gen,
    epochs=CONFIG['EPOCHS_PHASE2'],
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    verbose=1
)

phase2_time = time.time() - start_time
final_acc = max(history2.history['val_accuracy'])
final_top3 = max(history2.history['val_top3_acc'])

print(f"\n✅ Fase 2 completada en {phase2_time/60:.1f} minutos")
print(f"   Best Val Accuracy: {final_acc:.2%}")
print(f"   Best Val Top-3 Accuracy: {final_top3:.2%}")

total_time = phase1_time + phase2_time
print(f"\n⏱️ Tiempo total de entrenamiento: {total_time/60:.1f} minutos")

In [ ]:
# 8. Evaluar modelo y mostrar métricas detalladas
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Cargar mejor modelo
model.load_weights('best_phase2.keras')

print("=" * 60)
print("📊 EVALUACIÓN FINAL DEL MODELO")
print("=" * 60)

# Evaluar en validation set
val_gen.reset()
results = model.evaluate(val_gen, verbose=0)
print(f"\n🎯 Métricas finales:")
print(f"   Loss: {results[0]:.4f}")
print(f"   Accuracy: {results[1]:.2%}")
print(f"   Top-3 Accuracy: {results[2]:.2%}")

# Predicciones para análisis
val_gen.reset()
y_pred_probs = model.predict(val_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = val_gen.classes

# Análisis de confianza
confidence_scores = np.max(y_pred_probs, axis=1)
print(f"\n📈 Distribución de confianza:")
print(f"   Media: {np.mean(confidence_scores):.2%}")
print(f"   Mediana: {np.median(confidence_scores):.2%}")
print(f"   Min: {np.min(confidence_scores):.2%}")
print(f"   Max: {np.max(confidence_scores):.2%}")
print(f"   >50%: {np.mean(confidence_scores > 0.5):.1%} de predicciones")
print(f"   >70%: {np.mean(confidence_scores > 0.7):.1%} de predicciones")
print(f"   >90%: {np.mean(confidence_scores > 0.9):.1%} de predicciones")

# Graficar historial de entrenamiento
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy
axes[0].plot(history1.history['accuracy'], label='Train P1')
axes[0].plot(history1.history['val_accuracy'], label='Val P1')
offset = len(history1.history['accuracy'])
axes[0].plot(range(offset, offset+len(history2.history['accuracy'])), 
             history2.history['accuracy'], label='Train P2')
axes[0].plot(range(offset, offset+len(history2.history['val_accuracy'])), 
             history2.history['val_accuracy'], label='Val P2')
axes[0].axvline(x=offset-0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Accuracy por Epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history1.history['loss'], label='Train P1')
axes[1].plot(history1.history['val_loss'], label='Val P1')
axes[1].plot(range(offset, offset+len(history2.history['loss'])), 
             history2.history['loss'], label='Train P2')
axes[1].plot(range(offset, offset+len(history2.history['val_loss'])), 
             history2.history['val_loss'], label='Val P2')
axes[1].axvline(x=offset-0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('Loss por Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Distribución de confianza
axes[2].hist(confidence_scores, bins=50, edgecolor='black', alpha=0.7)
axes[2].axvline(x=np.mean(confidence_scores), color='red', linestyle='--', label=f'Media: {np.mean(confidence_scores):.2f}')
axes[2].set_title('Distribución de Confianza')
axes[2].set_xlabel('Confianza')
axes[2].set_ylabel('Frecuencia')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_metrics.png', dpi=150)
plt.show()

print("\n✅ Gráficas guardadas en 'training_metrics.png'")

In [ ]:
# 9. Guardar modelo en múltiples formatos
print("=" * 60)
print("💾 GUARDANDO MODELO")
print("=" * 60)

# Guardar en formato Keras
model.save('plant_disease_model.keras')
print("✅ Guardado: plant_disease_model.keras")

# Guardar en SavedModel para conversión
model.save('plant_disease_savedmodel', save_format='tf')
print("✅ Guardado: plant_disease_savedmodel/")

# Convertir a TFLite (formato más compatible con MindSpore)
print("\n🔄 Convirtiendo a TFLite...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float32]

tflite_model = converter.convert()

with open('plant_disease_model.tflite', 'wb') as f:
    f.write(tflite_model)

tflite_size = os.path.getsize('plant_disease_model.tflite') / (1024*1024)
print(f"✅ Guardado: plant_disease_model.tflite ({tflite_size:.1f} MB)")

# También exportar a ONNX
print("\n🔄 Convirtiendo a ONNX...")
!python -m tf2onnx.convert --saved-model plant_disease_savedmodel --output plant_disease_model.onnx --opset 13 2>/dev/null

if os.path.exists('plant_disease_model.onnx'):
    onnx_size = os.path.getsize('plant_disease_model.onnx') / (1024*1024)
    print(f"✅ Guardado: plant_disease_model.onnx ({onnx_size:.1f} MB)")
else:
    print("⚠️ ONNX no generado (usaremos TFLite)")

In [ ]:
# 10. Crear archivo de labels optimizado para Android
import json

class_names = sorted(train_gen.class_indices.keys())
labels = []

# Traducciones completas al español
translations = {
    'healthy': 'Saludable',
    'Early_blight': 'Tizón temprano',
    'Late_blight': 'Tizón tardío',
    'Bacterial_spot': 'Mancha bacteriana',
    'Leaf_Mold': 'Moho foliar',
    'Septoria_leaf_spot': 'Septoriosis',
    'Spider_mites Two-spotted_spider_mite': 'Ácaros',
    'Target_Spot': 'Mancha diana',
    'Tomato_Yellow_Leaf_Curl_Virus': 'Virus rizado amarillo',
    'Tomato_mosaic_virus': 'Virus del mosaico',
    'Powdery_mildew': 'Oídio',
    'Black_rot': 'Podredumbre negra',
    'Cedar_apple_rust': 'Roya del manzano',
    'Esca_(Black_Measles)': 'Esca',
    'Leaf_blight_(Isariopsis_Leaf_Spot)': 'Tizón foliar',
    'Common_rust_': 'Roya común',
    'Northern_Leaf_Blight': 'Tizón norteño',
    'Cercospora_leaf_spot Gray_leaf_spot': 'Mancha gris',
    'Haunglongbing_(Citrus_greening)': 'Huanglongbing',
    'Apple_scab': 'Sarna del manzano',
    'Leaf_scorch': 'Quemadura foliar',
}

for i, name in enumerate(class_names):
    parts = name.split('___')
    crop_raw = parts[0].replace('_', ' ').replace('(', '').replace(')', '')
    disease_raw = parts[1] if len(parts) > 1 else 'healthy'
    display = translations.get(disease_raw, disease_raw.replace('_', ' '))
    
    labels.append({
        "id": i,
        "name": name,
        "crop": crop_raw,
        "display": display
    })

# Guardar con formato esperado por Android
output = {"labels": labels}
with open('plant_disease_labels.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"✅ Labels guardados: {len(labels)} clases")
print("\n📋 Primeras 5 clases:")
for l in labels[:5]:
    print(f"   {l['id']:2d}: {l['crop']} → {l['display']}")
print("\n📋 Últimas 5 clases:")
for l in labels[-5:]:
    print(f"   {l['id']:2d}: {l['crop']} → {l['display']}")

In [ ]:
# 11. Resumen final y descarga
print("=" * 60)
print("🎉 ENTRENAMIENTO COMPLETADO")
print("=" * 60)

print(f"""
📊 RESULTADOS:
   • Accuracy: {final_acc:.2%}
   • Top-3 Accuracy: {final_top3:.2%}
   • Confianza media: {np.mean(confidence_scores):.2%}
   • Tiempo total: {total_time/60:.1f} minutos

📁 ARCHIVOS GENERADOS:
   • plant_disease_model.keras ({os.path.getsize('plant_disease_model.keras')/(1024*1024):.1f} MB)
   • plant_disease_model.tflite ({os.path.getsize('plant_disease_model.tflite')/(1024*1024):.1f} MB)
   • plant_disease_labels.json
   • training_metrics.png
""")

# Descargar archivos
from google.colab import files

print("📥 Descargando archivos...")
files.download('plant_disease_model.tflite')
files.download('plant_disease_labels.json')
files.download('training_metrics.png')

print("""
═══════════════════════════════════════════════════════════
📱 INSTRUCCIONES PARA ANDROID
═══════════════════════════════════════════════════════════

1. En tu PC Linux, descarga y extrae MindSpore Lite:
   wget https://ms-release.obs.cn-north-4.myhuaweicloud.com/2.3.1/MindSpore/lite/release/linux/x86_64/mindspore-lite-2.3.1-linux-x64.tar.gz
   tar -xzf mindspore-lite-2.3.1-linux-x64.tar.gz

2. Convierte TFLite a MindSpore:
   export LD_LIBRARY_PATH=mindspore-lite-2.3.1-linux-x64/tools/converter/lib:$LD_LIBRARY_PATH
   ./mindspore-lite-2.3.1-linux-x64/tools/converter/converter/converter_lite \\
       --fmk=TFLITE \\
       --modelFile=plant_disease_model.tflite \\
       --outputFile=plant_disease_model

3. Copia a tu proyecto Android:
   cp plant_disease_model.ms app/src/main/assets/
   cp plant_disease_labels.json app/src/main/assets/

4. Recompila la app:
   ./gradlew assembleDebug

═══════════════════════════════════════════════════════════
""")

# ⏱️ Comparación de Tiempos: T4 vs Colab Pro

## Estimación de tiempos de entrenamiento

| GPU | Colab Plan | Batch Size | Fase 1 (15 ep) | Fase 2 (25 ep) | **Total** | Costo |
|-----|------------|------------|----------------|----------------|-----------|-------|
| **T4** | Free | 32 | ~35-45 min | ~50-70 min | **85-115 min** | Gratis |
| **V100** | Pro ($10/mes) | 48 | ~15-20 min | ~25-35 min | **40-55 min** | ~$10/mes |
| **A100** | Pro+ ($50/mes) | 64 | ~8-12 min | ~15-20 min | **23-32 min** | ~$50/mes |

## 🚀 Mejora esperada con Colab Pro (V100)
- **Reducción de tiempo: ~50-55%** (de ~100 min a ~45 min)
- Batch size mayor = mejor convergencia
- Más RAM GPU = menos swapping
- Prioridad en cola de ejecución

## 🚀🚀 Mejora esperada con Colab Pro+ (A100)
- **Reducción de tiempo: ~70-75%** (de ~100 min a ~25-30 min)
- Tensor Cores optimizados para deep learning
- 40GB VRAM vs 16GB de T4
- Ideal para experimentación rápida

## Recomendación
Para **producción y experimentación frecuente**: **Colab Pro ($10/mes)** ofrece el mejor balance costo/beneficio.
- Entrenas 2-3x más rápido
- Puedes iterar más veces en el mismo tiempo
- Sessions más largas (24h vs 12h)